In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [2]:
# 21 : Jointures orders + customers

PATH = "/home/jovyan/data/tmp"

df_orders = spark.read.parquet(f"{PATH}/orders")
df_customers = spark.read.parquet(f"{PATH}/customers")

df_ord_cust = (
    df_orders
    .join(
        df_customers,
        on="customer_id",
        how = "left"
    ).select(
        "order_id",
        "company_name",
        "country",
        "order_date",
        "freight")
)

In [3]:
# 22 : Jointures order_details + products

df_order_details = spark.read.parquet(f"{PATH}/order_details")
df_products = spark.read.parquet(f"{PATH}/products")


df_ord_de_pro = (
    df_order_details
    .join(
        df_products,
        on="product_id",
        how = "left"
    ).select(
        "product_id",
        "category_id",
        "unit_price",
        "product_name")
)

In [4]:
# 23 : Jointures products + categories

df_categories = spark.read.parquet(f"{PATH}/categories")

df_prod_enrich = (
    df_products
    .join(
        df_categories,
        on="category_id",
        how = "left"
    )
)

In [5]:
# 24_A : DataFrame enrichi complet

df_employees = spark.read.parquet(f"{PATH}/employees")
df_shippers = spark.read.parquet(f"{PATH}/shippers")
df_suppliers = spark.read.parquet(f"{PATH}/suppliers")

df_complet = (
    df_order_details
    .join(df_orders,on="order_id")
    .join(df_customers, on="customer_id")
    .join(df_products, on="product_id")
    .join(df_categories, on="category_id")
    .join(df_employees, on="employee_id")
    .join(df_shippers, on="shipper_id")
)

from collections import Counter
counts = Counter(df_complet.columns)
colonnes_doublons = {col: count for col, count in counts.items() if count > 1}
print("Colonnes présentes en double :")
print(colonnes_doublons)

Colonnes présentes en double :
{'company_name': 2, 'city': 2, 'country': 2, 'phone': 2}


In [6]:
# 24_B : DataFrame enrichi complet

# Renomme la colonne company_name dans customers et shippers
df_customers = (
    df_customers
    .withColumnsRenamed({
        "company_name": "customers_company_name"
    })
)

df_shippers = (
    df_shippers
    .withColumnsRenamed({
        "company_name": "shippers_company_name"
    })
)

# Renomme la colonne city dans employees et customers

df_employees = (
    df_employees
    .withColumnsRenamed({
        "city": "employess_city"
    })
)

df_customers = (
    df_customers
    .withColumnsRenamed({
        "city": "customers_city"
    })
)

# Renomme la colonne country dans employees et customers

df_employees = (
    df_employees
    .withColumnsRenamed({
        "country": "employees_country"
    })
)

df_customers = (
    df_customers
    .withColumnsRenamed({
        "country": "customers_country"
    })
)

# Renomme la colonne phone dans shippers et customers

df_shippers = (
    df_shippers
    .withColumnsRenamed({
        "phone": "shippers_phone"
    })
)

df_customers = (
    df_customers
    .withColumnsRenamed({
        "phone": "customers_phone"
    })
)

In [7]:
df_orders_enriched = (
    df_order_details
    .join(df_orders,on="order_id")
    .join(df_customers, on="customer_id")
    .join(df_products, on="product_id")
    .join(df_categories, on="category_id")
    .join(df_employees, on="employee_id")
    .join(df_shippers, on="shipper_id")
)

from collections import Counter
counts_2 = Counter(df_orders_enriched.columns)
colonnes_doublons_2 = {col: count for col, count in counts_2.items() if count > 1}
print("Colonnes présentes en double :")
print(colonnes_doublons_2)

Colonnes présentes en double :
{}


In [8]:
# 25 : CA par client

df_ca_client = (
    df_orders_enriched
    .groupBy("customers_company_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy(
        F.col("ca").desc()
    )
    .limit(10)
)

df_ca_client.show()

+----------------------+--------+
|customers_company_name|      ca|
+----------------------+--------+
|            QUICK-Stop| 54194.9|
|          Ernst Handel| 44382.9|
|    Save-a-lot Markets| 42298.2|
|       MÃ¨re Paillarde| 25614.6|
|  Rattlesnake Canyo...| 17788.6|
|         Simons bistro|17482.15|
|  Hungry Owl All-Ni...|16472.75|
|       Folk och fÃ¤ HB| 13371.5|
|      HILARION-Abastos|12343.18|
|   Berglunds snabbkÃ¶p| 12285.6|
+----------------------+--------+



In [9]:
# 26 : CA par catégorie

df_ca_cat = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca"),
        F.countDistinct("product_id").alias("nb_products")
    )
    .orderBy(
        F.col("ca").desc()
    )
)

df_ca_cat.show()

+--------------+--------+-----------+
| category_name|      ca|nb_products|
+--------------+--------+-----------+
|Dairy Products|115890.8|          9|
|     Beverages| 95771.6|          9|
|   Confections|87227.77|         13|
|       Seafood|71320.65|         12|
|    Condiments|59273.35|         11|
|Grains/Cereals|54480.95|          6|
|       Produce|43108.55|          4|
|  Meat/Poultry|12405.65|          2|
+--------------+--------+-----------+



In [10]:
# 27 : CA par mois

df_ca_month = (
    df_orders_enriched
    .withColumn("mois", F.date_trunc("month", F.col("order_date")))
    .withColumn("year", F.date_trunc("year", F.col("order_date")))
    .groupBy("mois", "year")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy("mois", "year")
)

df_ca_month.show()

+-------------------+-------------------+--------+
|               mois|               year|      ca|
+-------------------+-------------------+--------+
|1997-01-01 00:00:00|1997-01-01 00:00:00| 55862.0|
|1997-02-01 00:00:00|1997-01-01 00:00:00| 33803.4|
|1997-03-01 00:00:00|1997-01-01 00:00:00| 34292.3|
|1997-04-01 00:00:00|1997-01-01 00:00:00|43754.05|
|1997-05-01 00:00:00|1997-01-01 00:00:00| 51668.5|
|1997-06-01 00:00:00|1997-01-01 00:00:00| 32210.7|
|1997-07-01 00:00:00|1997-01-01 00:00:00|49295.43|
|1997-08-01 00:00:00|1997-01-01 00:00:00| 40383.8|
|1997-09-01 00:00:00|1997-01-01 00:00:00|45746.68|
|1997-10-01 00:00:00|1997-01-01 00:00:00|51204.12|
|1997-11-01 00:00:00|1997-01-01 00:00:00|42134.86|
|1997-12-01 00:00:00|1997-01-01 00:00:00|59123.48|
+-------------------+-------------------+--------+



In [11]:
# 28 : Performance par employé

df_comm_employees = (
    df_orders_enriched
    .groupBy("full_name")
    .agg(
        F.countDistinct("order_id").alias("nb_comm_traitees"),
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca"),
        F.round(F.avg(F.datediff(F.col("shipped_date"), F.col("order_date"))), 1).alias("delai_moy_livraison")
    )
    .orderBy(
        F.col("ca").desc()
    )
)

df_comm_employees.show()

+----------------+----------------+---------+-------------------+
|       full_name|nb_comm_traitees|       ca|delai_moy_livraison|
+----------------+----------------+---------+-------------------+
|Margaret Peacock|              75|111988.38|                8.3|
| Janet Leverling|              71|100559.39|                8.9|
|   Nancy Davolio|              54| 85301.58|                7.8|
|   Andrew Fuller|              40|  58438.5|               10.2|
|     Robert King|              33|  55382.4|                9.8|
|  Laura Callahan|              53| 50276.67|                8.0|
|  Michael Suyama|              33|  36600.4|                7.9|
|  Anne Dodsworth|              18| 22517.15|               10.0|
| Steven Buchanan|              18| 18414.85|                6.5|
+----------------+----------------+---------+-------------------+



In [12]:
# 29 : Window functions - Rang

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window = Window.partitionBy("category_name").orderBy(F.col("ca").desc())

df_ca_rank = (
    df_orders_enriched
    .groupBy("category_name", "product_name")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .withColumn("dense_rank", F.dense_rank().over(window))
)

df_ca_rank.show()

+-------------+--------------------+-------+----------+
|category_name|        product_name|     ca|dense_rank|
+-------------+--------------------+-------+----------+
|    Beverages|      CÃ´te de Blaye|51962.2|         1|
|    Beverages|         Ipoh Coffee|11518.4|         2|
|    Beverages|      LakkalikÃ¶Ã¶ri| 7707.6|         3|
|    Beverages|       Outback Lager| 5880.0|         4|
|    Beverages|      Steeleye Stout| 5857.2|         5|
|    Beverages|    Chartreuse verte| 4928.4|         6|
|    Beverages|RhÃ¶nbrÃ¤u Kloste...| 4767.8|         7|
|    Beverages|       Sasquatch Ale| 2240.0|         8|
|    Beverages|Laughing Lumberja...|  910.0|         9|
|   Condiments|     Sirop d'Ã©rable|10539.3|         1|
|   Condiments|Louisiana Fiery H...| 9898.0|         2|
|   Condiments|        Vegie-spread| 7417.1|         3|
|   Condiments|        Gula Malacca|7082.05|         4|
|   Condiments|Chef Anton's Caju...| 5737.6|         5|
|   Condiments|Original Frankfur...| 5181.8|    

In [13]:
# 30 : Window functions - Cumul

window_2 = Window.orderBy("year", "mois")

df_ca_cumul = (
    df_ca_month
    .withColumn("ca_cumul", F.sum("ca").over(window_2))

)
df_ca_cumul.show()

+-------------------+-------------------+--------+------------------+
|               mois|               year|      ca|          ca_cumul|
+-------------------+-------------------+--------+------------------+
|1997-01-01 00:00:00|1997-01-01 00:00:00| 55862.0|           55862.0|
|1997-02-01 00:00:00|1997-01-01 00:00:00| 33803.4|           89665.4|
|1997-03-01 00:00:00|1997-01-01 00:00:00| 34292.3|          123957.7|
|1997-04-01 00:00:00|1997-01-01 00:00:00|43754.05|         167711.75|
|1997-05-01 00:00:00|1997-01-01 00:00:00| 51668.5|         219380.25|
|1997-06-01 00:00:00|1997-01-01 00:00:00| 32210.7|         251590.95|
|1997-07-01 00:00:00|1997-01-01 00:00:00|49295.43|         300886.38|
|1997-08-01 00:00:00|1997-01-01 00:00:00| 40383.8|         341270.18|
|1997-09-01 00:00:00|1997-01-01 00:00:00|45746.68|         387016.86|
|1997-10-01 00:00:00|1997-01-01 00:00:00|51204.12|         438220.98|
|1997-11-01 00:00:00|1997-01-01 00:00:00|42134.86|480355.83999999997|
|1997-12-01 00:00:00

In [14]:
# 31 : Tri et limites

product_vendu = (
    df_orders_enriched
    .groupBy("product_id", "product_name")
    .agg(
        F.sum(F.col("quantite")).alias("nb_prod_vendu")
    )
    .orderBy(F.col("nb_prod_vendu").desc())
    .limit(5)
)

product_vendu.show()

+----------+--------------------+-------------+
|product_id|        product_name|nb_prod_vendu|
+----------+--------------------+-------------+
|        56|Gnocchi di nonna ...|          971|
|        59|Raclette Courdavault|          752|
|        60|   Camembert Pierrot|          665|
|        75|RhÃ¶nbrÃ¤u Kloste...|          630|
|        21| Sir Rodney's Scones|          610|
+----------+--------------------+-------------+



In [15]:
customer_country = (
    df_orders_enriched
    .groupBy("customers_country")
    .agg(
        F.round(F.sum(F.col("prix_unitaire") * F.col("quantite")), 2).alias("ca")
    )
    .orderBy(F.col("ca").desc())
    .limit(3)
)

customer_country.show()

+-----------------+---------+
|customers_country|       ca|
+-----------------+---------+
|          GERMANY|106411.65|
|              USA| 95563.42|
|          AUSTRIA|  51080.0|
+-----------------+---------+



In [16]:
# 32 : Ecriture en parquet 

PATH_2 = "/home/jovyan/data/output"

df_orders_enriched.write.mode("overwrite").parquet(f"{PATH_2}/df_orders_enriched")

### Notebook 4 -- Ecriture Parquet (et chargement PostgreSQL bonus)

In [17]:
# 33 : Relire le parquet

df_orders_enriched_2 = spark.read.parquet(f"{PATH_2}/df_orders_enriched")

print(f"nombre de lignes après le parquet : {df_orders_enriched_2.count()}")

nombre de lignes après le parquet : 893


In [18]:
print(f"nombre de lignes avant le parquet : {df_orders_enriched.count()}")

nombre de lignes avant le parquet : 893


In [19]:
df_orders_enriched_2.printSchema()

root
 |-- shipper_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: string (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: string (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customers_company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- con

In [20]:
# 34 : Comparer CSV vs Parquet

# Le fichier Parquet est plus léger que les CSV

In [24]:
# 35 : Partitionnement 

df_orders_enriched.write.mode("overwrite").partitionBy("customers_country").parquet(f"{PATH_2}/df_orders_enriched_countryc")

# Spark a créé un dossier pour un pays. Ces dossiers contiennent chacun un fichier .parquet. 

In [25]:
# 36 : Chargement PostgreSQL via JDBC


jdbc_url = "jdbc:postgresql://postgres:5432/tradecorp"

(
    df_orders_enriched
    .write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "orders_enriched")
    .option("user", "tradecorp")
    .option("password", "tradecorp")
    .option("driver", "org.postgresql.Driver")
    .mode("overwrite")
    .save()
)



In [23]:
# 37 : Vérifier dans pgAdmin

# J'ai essayé sur Dbeaver, cela fonctionne